# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [ ]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").first()[0])
spark.sql("SELECT * FROM v_silver WHERE cancelled = 0 AND arrival_delay IS NOT NULL").createOrReplaceTempView("v_observed")
print("Les analyses de retard portent sur les vols non annulés avec retard à l'arrivée connu.")

## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [ ]:
df_dist_summary = spark.sql("""
SELECT COUNT(*) AS n_vols, AVG(arrival_delay) AS moyenne,
       STDDEV(arrival_delay) AS ecart_type, SKEWNESS(arrival_delay) AS asymetrie,
       PERCENTILE_APPROX(arrival_delay, 0.25, 10000) AS p25,
       PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane,
       PERCENTILE_APPROX(arrival_delay, 0.75, 10000) AS p75,
       PERCENTILE_APPROX(arrival_delay, 0.90, 10000) AS p90,
       PERCENTILE_APPROX(arrival_delay, 0.95, 10000) AS p95,
       MAX(arrival_delay) AS maximum
FROM v_observed
""")
df_dist_summary.show(truncate=False)
distribution = df_dist_summary.first()
if distribution.n_vols < 2 or distribution.asymetrie is None:
    raise ValueError("Distribution insuffisante pour interpréter l'asymétrie.")
print(f"Moyenne : {distribution.moyenne:.2f} min ; médiane : {distribution.mediane} min ; IQR : {distribution.p75 - distribution.p25} min ; asymétrie : {distribution.asymetrie:.2f}.")

**Justification statistique :** une asymétrie positive et une moyenne supérieure à la médiane indiquent que les retards élevés tirent la moyenne vers le haut. La médiane et l'IQR décrivent le vol typique et sa dispersion sans être dominés par les extrêmes ; P90/P95 rendent visibles les retards importants. Les résultats exacts sont affichés ci-dessus.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Ici, le Pareto porte sur la somme des **minutes de retard positives** : une arrivée en avance contribue zéro. Ce choix évite que les avances annulent les retards et conserve un cumul croissant. Les vols annulés et les retards inconnus sont exclus de ce calcul.

In [ ]:
GOLD_PARETO_PATH = "/lakehouse/gold/flights_airline_pareto"
df_gold_1 = spark.sql("""
WITH grouped AS (
  SELECT airline, MAX(airline_name) AS airline_name, COUNT(*) AS n_vols,
         SUM(GREATEST(arrival_delay, 0)) AS minutes_retard,
         AVG(arrival_delay) AS moyenne, PERCENTILE_APPROX(arrival_delay, 0.25, 10000) AS p25,
         PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane,
         PERCENTILE_APPROX(arrival_delay, 0.75, 10000) AS p75,
         PERCENTILE_APPROX(arrival_delay, 0.90, 10000) AS p90,
         CAST(100 AS DOUBLE) * AVG(CASE WHEN arrival_delay >= 15 THEN CAST(1 AS DOUBLE) ELSE CAST(0 AS DOUBLE) END) AS retard_pct
  FROM v_observed GROUP BY airline
), ranked AS (
  SELECT *, p75 - p25 AS iqr,
         DENSE_RANK() OVER (ORDER BY minutes_retard DESC) AS rang,
         SUM(minutes_retard) OVER (ORDER BY minutes_retard DESC, airline ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumul,
         SUM(minutes_retard) OVER () AS total
  FROM grouped
)
SELECT airline, airline_name, n_vols, minutes_retard, moyenne, p25, mediane, p75, p90, iqr, retard_pct, rang,
       CASE WHEN total > 0 THEN CAST(100 AS DOUBLE) * cumul / total END AS pareto_pct
FROM ranked
""")
df_gold_1.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PARETO_PATH)
df_gold_1.orderBy("rang", "airline").show(15, truncate=False)

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [ ]:
GOLD_TIME_PATH = "/lakehouse/gold/flights_monthly"
df_gold_2 = spark.sql("""
WITH monthly AS (
  SELECT DATE_TRUNC('MONTH', flight_date) AS mois, COUNT(*) AS n_vols,
         AVG(arrival_delay) AS moyenne,
         PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane,
         CAST(100 AS DOUBLE) * AVG(CASE WHEN arrival_delay >= 15 THEN CAST(1 AS DOUBLE) ELSE CAST(0 AS DOUBLE) END) AS retard_pct
  FROM v_observed GROUP BY DATE_TRUNC('MONTH', flight_date)
), previous AS (
  SELECT *, LAG(n_vols) OVER (ORDER BY mois) AS vols_mois_precedent FROM monthly
)
SELECT *, CASE WHEN vols_mois_precedent > 0 THEN CAST(100 AS DOUBLE) * (n_vols - vols_mois_precedent) / vols_mois_precedent END AS evolution_vols_pct
FROM previous
""")
df_gold_2.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_TIME_PATH)
df_gold_2.orderBy("mois").show(20, truncate=False)

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'`. Les nombreux petits fichiers augmentent les métadonnées gérées par le NameNode et le nombre de tâches ; la compaction regroupe les fichiers sans changer les lignes. Les anciennes versions restent disponibles tant qu'elles ne sont pas supprimées par une opération de maintenance.

In [ ]:
GOLD_AIRPORT_PATH = "/lakehouse/gold/flights_airline_top_origin"
df_gold_3 = spark.sql("""
WITH grouped AS (
  SELECT airline, origin_airport, MAX(origin_airport_name) AS airport_name,
         COUNT(*) AS n_vols, PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane
  FROM v_observed GROUP BY airline, origin_airport
), ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY airline ORDER BY n_vols DESC, origin_airport) AS position
  FROM grouped
)
SELECT * FROM ranked WHERE position = 1
""")
df_gold_3.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_AIRPORT_PATH)
df_gold_3.orderBy("airline").show(20, truncate=False)

silver_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
before_files = silver_delta.detail().select("numFiles").first()[0]
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", str(128 * 1024 * 1024))
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
after_files = silver_delta.detail().select("numFiles").first()[0]
print(f"Fichiers Silver actifs : {before_files} avant, {after_files} après OPTIMIZE.")